# LSTM model using stock market prices

In this notebook, I will implement an LSTM model and train it based on stock market prices using data I get from Nasdaq.

- META: https://www.nasdaq.com/market-activity/stocks/meta/historical?page=1&rows_per_page=10&timeline=y10
- AMZN: https://www.nasdaq.com/market-activity/stocks/amzn/historical?page=1&rows_per_page=10&timeline=y10
- AAPL: https://www.nasdaq.com/market-activity/stocks/aapl/historical?page=1&rows_per_page=10&timeline=y10
- NFLX: https://www.nasdaq.com/market-activity/stocks/nflx/historical?page=1&rows_per_page=10&timeline=y10
- GOOGL: https://www.nasdaq.com/market-activity/stocks/googl/historical?page=1&rows_per_page=10&timeline=y10
- TSLA: https://www.nasdaq.com/market-activity/stocks/tsla/historical?page=1&rows_per_page=10&timeline=y10
- MSFT: https://www.nasdaq.com/market-activity/stocks/msft/historical?page=1&rows_per_page=10&timeline=y10

Although I have multiple data collected here, I will just focus on one data set (META) right now. Maybe I can do something later with more data sets.

Also, for this notebook, I will try to use the plotly package.

### Importing Packages and Libraries

In [27]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

### Data Collection

In [28]:
df_META = pd.read_csv("data/META.csv")
df_AMZN = pd.read_csv("data/AMZN.csv")
df_AAPL = pd.read_csv("data/AAPL.csv")
df_NFLX = pd.read_csv("data/NFLX.csv")
df_GOOGL = pd.read_csv("data/GOOGL.csv")
df_TSLA = pd.read_csv("data/TSLA.csv")
df_MSFT = pd.read_csv("data/MSFT.csv")

print(df_META.shape)
df_META.head()

(2514, 6)


,Date,Close/Last,Volume,Open,High,Low
0,07/03/2025,$719.01,8601653,$726.61,$729.03,$714.42
1,07/02/2025,$713.57,9336740,$715.325,$720.30,$712.80
2,07/01/2025,$719.22,13431250,$736.875,$737.7499,$715.37
3,06/30/2025,$738.09,15402110,$744.55,$747.90,$734.25
4,06/27/2025,$733.63,18775740,$726.515,$735.43,$725.86


### Data Cleaning/Wrangling

In [29]:
# df_META = df_META.rename(columns = {'Close/Last' : 'Close'})
# df_AMZN = df_AMZN.rename(columns = {'Close/Last' : 'Close'})
# df_AAPL = df_AAPL.rename(columns = {'Close/Last' : 'Close'})
# df_NFLX = df_NFLX.rename(columns = {'Close/Last' : 'Close'})
# df_GOOGL = df_GOOGL.rename(columns = {'Close/Last' : 'Close'})
# df_MSFT = df_MSFT.rename(columns = {'Close/Last' : 'Close'})

df = df_TSLA

df['Date'] = pd.to_datetime(df['Date'], format="%m/%d/%Y")
df = df.sort_values('Date')

df['Close'] = df['Close/Last'].replace({'\$':''}, regex=True).astype(float)
df['Open'] = df['Open'].replace({'\$':''}, regex=True).astype(float)
df['High'] = df['High'].replace({'\$':''}, regex=True).astype(float)
df['Low'] = df['Low'].replace({'\$':''}, regex=True).astype(float)

<>:13: SyntaxWarning:

invalid escape sequence '\$'

<>:14: SyntaxWarning:

invalid escape sequence '\$'

<>:15: SyntaxWarning:

invalid escape sequence '\$'

<>:16: SyntaxWarning:

invalid escape sequence '\$'

<>:13: SyntaxWarning:

invalid escape sequence '\$'

<>:14: SyntaxWarning:

invalid escape sequence '\$'

<>:15: SyntaxWarning:

invalid escape sequence '\$'

<>:16: SyntaxWarning:

invalid escape sequence '\$'

C:\Users\danch\AppData\Local\Temp\ipykernel_13904\1771279860.py:13: SyntaxWarning:

invalid escape sequence '\$'

C:\Users\danch\AppData\Local\Temp\ipykernel_13904\1771279860.py:14: SyntaxWarning:

invalid escape sequence '\$'

C:\Users\danch\AppData\Local\Temp\ipykernel_13904\1771279860.py:15: SyntaxWarning:

invalid escape sequence '\$'

C:\Users\danch\AppData\Local\Temp\ipykernel_13904\1771279860.py:16: SyntaxWarning:

invalid escape sequence '\$'



### Data Visualization

In [30]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Date'], y=df['Close'], name='Close', marker={'color':'yellow'}, mode='lines'))
fig.add_trace(go.Scatter(x=df['Date'], y=df['High'], name='High', marker={'color':'green'}, mode='lines'))
fig.add_trace(go.Scatter(x=df['Date'], y=df['Low'], name='Low', marker={'color':'red'}, mode='lines'))

fig.update_layout(title="TSLA stock prices",
                  xaxis_title="Date",
                  yaxis_title="Stock price")

fig.show()


### Create and Split Data sequences for LSTM
We need to create sequences to train the LSTM. Sequences are needed since they affect one after the other.

In [31]:
prices = df['Close'].values.reshape(-1, 1)

# Transform the prices
scaler = MinMaxScaler()
prices_scaled = scaler.fit_transform(prices)

def create_sequences(data, seq_length):
    xs = []
    ys = []
    for i in range(len(data) - seq_length):
        x = data[i:(i+seq_length)]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LEN = 45
X, y = create_sequences(prices_scaled, SEQ_LEN)

# Train/test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

### Pytorch Custom Dataset
Since we are working with pytorch, we need to convert them to tensors and pytorch's Dataset (pytorch.utils.Dataset), and use DataLoader

In [32]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = StockDataset(X_train, y_train)
test_ds = StockDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

### LSTM model initialization

In [33]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # Take last output
        out = self.fc(out)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = LSTMModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

### Model Training

In [34]:
EPOCHS = 100
train_losses = []
for epoch in range(1, EPOCHS+1):
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        xb = xb.view(xb.size(0), xb.size(1), 1)  # (batch, seq, 1)

        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        epoch_loss = epoch_loss + loss.item()*xb.size(0)
    
    avg_loss = epoch_loss / len(train_loader.dataset)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch}/{EPOCHS}, Training Loss: {avg_loss:.6f}")

Epoch 1/100, Training Loss: 0.056016
Epoch 2/100, Training Loss: 0.094507
Epoch 3/100, Training Loss: 0.042180
Epoch 4/100, Training Loss: 0.038025
Epoch 5/100, Training Loss: 0.005580
Epoch 6/100, Training Loss: 0.001911
Epoch 7/100, Training Loss: 0.001090
Epoch 8/100, Training Loss: 0.001410
Epoch 9/100, Training Loss: 0.001797
Epoch 10/100, Training Loss: 0.002283
Epoch 11/100, Training Loss: 0.002746
Epoch 12/100, Training Loss: 0.002979
Epoch 13/100, Training Loss: 0.002920
Epoch 14/100, Training Loss: 0.002698
Epoch 15/100, Training Loss: 0.002459
Epoch 16/100, Training Loss: 0.002277
Epoch 17/100, Training Loss: 0.002161
Epoch 18/100, Training Loss: 0.002094
Epoch 19/100, Training Loss: 0.002055
Epoch 20/100, Training Loss: 0.002029
Epoch 21/100, Training Loss: 0.002005
Epoch 22/100, Training Loss: 0.001978
Epoch 23/100, Training Loss: 0.001945
Epoch 24/100, Training Loss: 0.001909
Epoch 25/100, Training Loss: 0.001870
Epoch 26/100, Training Loss: 0.001830
Epoch 27/100, Trainin

In [35]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(1, EPOCHS+1),
    y=train_losses,
    mode='lines'
))

fig.update_layout(
    title='Training Losses',
    xaxis_title='Training Loss',
    yaxis_title='Epochs',
    legend=dict(x=0, y=1),
    width=1400,
    height=500
)

fig.show()

### Model Evaluation

In [36]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    X_test_tensor = X_test_tensor.view(X_test_tensor.size(0), X_test_tensor.size(1), 1)
    preds = model(X_test_tensor).cpu().numpy()
    preds_inv = scaler.inverse_transform(preds)
    y_test_inv = scaler.inverse_transform(y_test)

r2 = r2_score(y_test_inv, preds_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, preds_inv))
print(f"R^2: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

R^2: 0.9365
RMSE: 17.3216


### Model Evaluation Visualizations

In [37]:
fig = go.Figure()

# Training prices
fig.add_trace(go.Scatter(
    x=df['Date'][:split+SEQ_LEN],
    y=scaler.inverse_transform(prices_scaled[:split+SEQ_LEN]).flatten(),
    mode='lines',
    name='Train Prices'
))

# Test prices
fig.add_trace(go.Scatter(
    x=df['Date'][split+SEQ_LEN:],
    y=scaler.inverse_transform(prices_scaled[split+SEQ_LEN:]).flatten(),
    mode='lines',
    name='Test Prices'
))

# Predicted test prices
fig.add_trace(go.Scatter(
    x=df['Date'][split+SEQ_LEN:],
    y=preds_inv.flatten(),
    mode='lines',
    name='Predicted Test Prices'
))

fig.update_layout(
    title='TSLA Predicted Stock Price',
    xaxis_title='Date',
    yaxis_title='Price',
    legend=dict(x=0, y=1),
    width=1400,
    height=500
)

fig.show()

### Forecasting Future Values

In [38]:
import matplotlib.patches as mpatches

plt.figure(figsize=(16,6))

# Training prices
# plt.plot(df['Date'][:split+SEQ_LEN], scaler.inverse_transform(prices_scaled[:split+SEQ_LEN]), color='blue', label='Train Prices')

# Test prices
# plt.plot(df['Date'][split+SEQ_LEN:split+SEQ_LEN+len(y_test)], scaler.inverse_transform(prices_scaled[split+SEQ_LEN:split+SEQ_LEN+len(y_test)]), color='orange', label='Test Prices')

# Predicted test prices
# plt.plot(df['Date'][split+SEQ_LEN:split+SEQ_LEN+len(preds_inv)], preds_inv, color='green', label='Predicted Test Prices')

# Predict future prices after the test set
future_steps = 120 # Number of days
last_seq = X_test[-1]
future_preds = []
current_seq = last_seq.copy()
for _ in range(future_steps):
    input_tensor = torch.tensor(current_seq, dtype=torch.float32).unsqueeze(0).to(device)
    input_tensor = input_tensor.view(1, input_tensor.size(1), 1)
    with torch.no_grad():
        next_pred = model(input_tensor).cpu().numpy().flatten()
    future_preds.append(next_pred[0])
    # Update sequence: drop first, append new prediction
    current_seq = np.roll(current_seq, -1)
    current_seq[-1] = next_pred


future_preds_inv = scaler.inverse_transform(np.array(future_preds).reshape(-1,1)).flatten()

# Future data
last_date = pd.to_datetime(df['Date'].iloc[split+SEQ_LEN+len(y_test)-1])
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=future_steps, freq='B')  # 'B' for business days


fig = go.Figure()

# Test prices
test_trace = go.Scatter(
    x=df['Date'].iloc[split+SEQ_LEN:split+SEQ_LEN+len(y_test)],
    y=scaler.inverse_transform(prices_scaled[split+SEQ_LEN:split+SEQ_LEN+len(y_test)]).flatten(),
    mode='lines',
    name='Test Prices',
    line=dict(color='orange')
)

# Predicted test prices
pred_trace = go.Scatter(
    x=df['Date'].iloc[split+SEQ_LEN:split+SEQ_LEN+len(preds_inv)],
    y=preds_inv.flatten() if hasattr(preds_inv, "flatten") else preds_inv,
    mode='lines',
    name='Predicted Test Prices',
    line=dict(color='green')
)

# Future predicted prices
future_trace = go.Scatter(
    x=future_dates,
    y=future_preds_inv.flatten() if hasattr(future_preds_inv, "flatten") else future_preds_inv,
    mode='lines',
    name='Future Predicted Prices',
    line=dict(color='red', dash='dash')
)

# Vertical line for split
split_idx = split+SEQ_LEN+len(y_test)-1
split_date = df['Date'].iloc[split_idx]
all_y = np.concatenate([
    scaler.inverse_transform(prices_scaled).flatten(),
    preds_inv.flatten() if hasattr(preds_inv, "flatten") else preds_inv,
    future_preds_inv.flatten() if hasattr(future_preds_inv, "flatten") else future_preds_inv
])
vline = go.Scatter(
    x=[split_date, split_date],
    y=[np.min(all_y), np.max(all_y)],
    mode='lines',
    name='Test/Future Split',
    line=dict(color='gray', dash='dot'),
    showlegend=False
)

layout = go.Layout(
    title='TSLA Stock Price Prediction: Test, Predicted, and Future',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Price'),
    legend=dict(x=0, y=1),
    width=1600,
    height=600
)

fig = go.Figure(data=[test_trace, pred_trace, future_trace, vline], layout=layout)
fig.show()


<Figure size 1600x600 with 0 Axes>

### Questions that got me thinking:
So as I was training this model over and over again, I noticed that the future predictions always have a downward trend, no matter how good the R^2 value gets, so I kept thinking, "Why is it always going downwards?? The trend has always been generally positive!"

After thinking some more and Searching online to understand LSTMs on a deeper level, I realize this:
- LSTMs use local sequences: they look back at each sequence and use those outputs as inputs (aka recurrent). There are small errors in each training and testing outputs which accumulate and cause a reverse output in general. As we can see in the graph above, the future prices slowly descend faster.
- LSTMs are known to model nonlinear tends, so because of this, the depend more on patterns found in each sequence.

I tried tuning the model's parameters such as number of hidden units, dropout, learning rate, and weight decay, yet it caused the R^2 value to drop and make the future predictions even flatline (lol). In a future model, I will look into something that is more capable at more long term extrapolations! Wish me luck hehe